# SafeTV — Cloud GPU Deep Action Recognition Fine-Tuning

This notebook trains and fine-tunes the **SafeTV R(2+1)D-18 Spatiotemporal Action Recognition Model** on surveillance violence datasets (**RWF-2000**, **RLVS**, or custom CCTV footage) using free NVIDIA GPUs (T4 / P100 / A100) on **Google Colab** or **Kaggle**.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 2. Clone SafeTV Repository & Install Dependencies

In [ ]:
!git clone https://github.com/eklavya434/safetv.git
%cd safetv
!pip install -r requirements.txt

## 3. Prepare or Ingest Violence Dataset
You can build the sample benchmark dataset or ingest extracted RWF-2000 / RLVS footage:

In [ ]:
# Build dataset with leak-free train/val/test splits (split strictly by source video)
!python prepare_dataset.py build-sample --num-clips 40
!python prepare_dataset.py info

## 4. Run GPU Fine-Tuning
Fine-tunes layer3, layer4, and the SafeTV classification head using mixed precision and Cosine Annealing learning rate scheduling:

In [ ]:
!python train.py --epochs 15 --batch-size 8 --lr 1e-3 --backbone-lr 5e-5 --device cuda

## 5. Evaluate on Unseen Holdout Test Videos

In [ ]:
!python train.py --evaluate-only --device cuda

## 6. Inspect Saved Checkpoint & Download Weights

In [ ]:
!ls -lh models/violence_detector/
import json
with open('models/violence_detector/training_metadata.json') as f:
    meta = json.load(f)
print('Best Val F1:', meta['best_val_metrics']['f1_score'])
print('Test Accuracy:', meta['unseen_test_evaluation']['accuracy'])